# 01_data_preparation.ipynb

In [ ]:
# ==========================================================
# 📦 SETUP
# ==========================================================
# Install dependencies if not yet installed
%pip install tensorflow pandas pillow tqdm

import os
import string
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image
from pickle import dump

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

print("✅ All libraries loaded.")

In [ ]:
# ==========================================================
# 📁 LOAD AND CLEAN CAPTIONS (Robust and Shareable)
# ==========================================================

import os
import string

# ✅ Auto-locate and set project root (if needed)
def find_project_root():
    current_path = os.getcwd()
    while True:
        if 'data' in os.listdir(current_path):
            return current_path
        parent = os.path.dirname(current_path)
        if parent == current_path:
            raise FileNotFoundError("❌ Could not find project root with 'data/' folder.")
        current_path = parent

# 🔁 Set working directory to project root
project_root = find_project_root()
os.chdir(project_root)
print("✅ Project root set to:", project_root)

# ✅ Define file paths
FOLDER_IMAGES = os.path.join("data", "raw", "images")
CAPTIONS_FILE = os.path.join("data", "raw", "captions.txt")

# ✅ Check if file exists
if not os.path.exists(CAPTIONS_FILE):
    raise FileNotFoundError(f"❌ File not found: {CAPTIONS_FILE}")
print("📄 Captions file found ✅")

# ==========================================================
# 🧠 Caption loader (robust against bad lines)
# ==========================================================

def load_captions(captions_file):
    captions_mapping = {}
    skipped_lines = 0

    with open(captions_file, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f.readlines(), 1):
            if '\t' not in line:
                skipped_lines += 1
                print(f"⚠️ Skipping malformed line {line_num}: {line.strip()}")
                continue

            img_name, caption = line.strip().split('\t')
            img_name = img_name.split('#')[0]
            caption = caption.lower().translate(str.maketrans('', '', string.punctuation))

            if img_name not in captions_mapping:
                captions_mapping[img_name] = []
            captions_mapping[img_name].append(caption)

    print(f"✅ Captions loaded. Total images: {len(captions_mapping)}")
    if skipped_lines:
        print(f"⚠️ Skipped {skipped_lines} malformed lines.")

    return captions_mapping

# ✅ Load captions dictionary
captions_dict = load_captions(CAPTIONS_FILE)


In [21]:
# ==========================================================
# 1. Dataset Handling (Flickr8k version of MS COCO step)
# ==========================================================

import os
from tqdm import tqdm
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.inception_v3 import preprocess_input
import numpy as np

# ✅ Paths
IMAGE_FOLDER = os.path.join("data", "raw", "images")
PROCESSED_IMAGE_DIR = os.path.join("data", "processed", "resized_images")

# ✅ Create processed image folder (if saving)
os.makedirs(PROCESSED_IMAGE_DIR, exist_ok=True)

# ✅ Preprocess and save resized/normalized image arrays
def preprocess_and_save_images(image_dir, target_dir):
    for image_name in tqdm(os.listdir(image_dir)):
        img_path = os.path.join(image_dir, image_name)
        try:
            img = load_img(img_path, target_size=(299, 299))
            img_array = img_to_array(img)
            img_array = preprocess_input(img_array)  # Normalize for InceptionV3
            # Optional: Save image as .npy
            np.save(os.path.join(target_dir, image_name.split('.')[0] + '.npy'), img_array)
        except Exception as e:
            print(f"⚠️ Failed to process {image_name}: {e}")

# ✅ Run it
preprocess_and_save_images(IMAGE_FOLDER, PROCESSED_IMAGE_DIR)
print("✅ All images resized and normalized.")


100%|██████████| 8091/8091 [03:47<00:00, 35.50it/s]

✅ All images resized and normalized.
